# LinQ Safety: обучение людей, касок и жилетов

Самодостаточный notebook. В Kaggle включите GPU и Internet. Прикрепите
`linq-ppe-training-data.zip` как приватный Dataset: Kaggle может распаковать его
самостоятельно. Notebook найдёт `manifest.jsonl`. Без готового корпуса он скачает
открытые источники и подготовит данные сам. Ключи и пароли не нужны.

Сначала `MODE="smoke"`: один цикл проверяет тракт. Затем новая полная сессия с
`MODE="full"`. Сохранённые last-checkpoints позволяют продолжать прерванный запуск.
При переносе outputs предыдущей сессии укажите `RESUME_FROM`.

Архитектура: YOLO12m для людей и двух задач СИЗ, ResNet152 для головы и торса.
Разные задачи имеют разные датасеты: отсутствие разметки жилета не равно его отсутствию.
Данные уже очищаются по SHA-256/pHash, исходным split и Roboflow-группам.
Отдельно нужны проверка разметки человеком и независимые реальные видео с зеркалами.
Не выдавайте качество на crop за event F1. Метрики >0.95 здесь не предзаписаны.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet',
                'ultralytics==8.4.88','onnx==1.16.2','onnxruntime==1.29.0',
                'gdown==5.2.0','pyyaml==6.0.3'],check=True)
from pathlib import Path
import json, shutil, zipfile, numpy as np, torch
WORK=Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
assert torch.cuda.is_available(), 'Выберите GPU в Settings -> Accelerator и перезапустите сессию.'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/2**30, 'GiB')

In [ ]:
SCRIPTS=WORK/'scripts'; SCRIPTS.mkdir(exist_ok=True)
(SCRIPTS/'download_ppe_data.py').write_text('"""Download public training sources; preserve provenance and actual counts.\n\nSH17 is optional (about 14 GB upstream). No credentials are stored. Archives\nare data only; never run Python from a dataset archive.\n"""\nimport argparse\nimport concurrent.futures\nimport hashlib\nimport json\nfrom pathlib import Path\nimport urllib.request\nimport zipfile\n\nSOURCES = {\n    \'construction\': {\'url\':\'https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip\',\n                     \'license\':\'AGPL-3.0\',\'page\':\'https://docs.ultralytics.com/datasets/detect/construction-ppe\'},\n    \'site_safety\': {\'url\':\'https://www.kaggle.com/api/v1/datasets/download/snehilsanyal/construction-site-safety-image-dataset-roboflow\',\n                    \'license\':\'CC BY 4.0 (source declaration)\',\'page\':\'https://www.kaggle.com/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow\'},\n    \'shwd\': {\'url\':\'https://drive.google.com/uc?id=1qWm7rrwvjAWs1slymbrLaCf7Q-wnGLEX\',\n             \'license\':\'MIT repository; includes SCUT-HEAD-derived images; retain source notices\',\n             \'page\':\'https://github.com/njvisionpower/Safety-Helmet-Wearing-Dataset\'},\n    \'sh17\': {\'url\':\'https://www.kaggle.com/api/v1/datasets/download/mugheesahmad/sh17-dataset-for-ppe-detection\',\n             \'license\':\'CC BY-NC-SA 4.0\',\'page\':\'https://github.com/ahmadmughees/SH17dataset\'},\n    \'hardhat_vest\': {\'url\':\'https://www.kaggle.com/api/v1/datasets/download/muhammetzahitaydn/hardhat-vest-dataset-v3\',\n                     \'license\':\'Check archive source notices; Kaggle declaration alone is not a provenance audit\',\n                     \'page\':\'https://www.kaggle.com/datasets/muhammetzahitaydn/hardhat-vest-dataset-v3\'},\n}\nEXPECTED_SHA256={\n    \'construction\':\'bef8dcb599aa4e9d9f5e602cb6fa7143d3c84d7f6a0ff40463d7f2a4c2632ccc\',\n    \'site_safety\':\'f1acf02c7116a09cdbb2b3ace1ee9b597977c0eb0bd0025e8949f81091fb2a3d\',\n    \'shwd\':\'2f6c5c347b778271bc8b27c2d61b731fc5fb8cdb4f45fce583dd2f82e6a46401\',\n    \'hardhat_vest\':\'61048c7f9a52c8672e1989e338e61781515056ae3809c2fa7a40477ddf20afaf\',\n}\n\n\ndef download_source(name,root):\n    spec=SOURCES[name];root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    target=root/(name+\'.zip\');partial=target.with_suffix(\'.zip.part\')\n    if not target.exists():\n        if name==\'shwd\':\n            import gdown\n            if not gdown.download(spec[\'url\'],str(partial),quiet=True):raise RuntimeError(\'SHWD download failed\')\n        else:\n            offset=partial.stat().st_size if partial.exists() else 0\n            headers={\'User-Agent\':\'LinQSafety dataset preparation/1.0\'}\n            if offset:headers[\'Range\']=f\'bytes={offset}-\'\n            request=urllib.request.Request(spec[\'url\'],headers=headers)\n            with urllib.request.urlopen(request,timeout=120) as response:\n                mode=\'ab\' if response.status==206 and offset else \'wb\'\n                with partial.open(mode) as stream:\n                    while chunk:=response.read(1024*1024):stream.write(chunk)\n        if not zipfile.is_zipfile(partial):raise ValueError(f\'{name}: response is not a ZIP dataset\')\n        partial.replace(target)\n    digest=hashlib.sha256()\n    with target.open(\'rb\') as stream:\n        while chunk:=stream.read(8*1024*1024):digest.update(chunk)\n    if name in EXPECTED_SHA256 and digest.hexdigest()!=EXPECTED_SHA256[name]:\n        raise ValueError(f\'{name}: source archive changed; review its version before using new data\')\n    folder=root/name;marker=folder/\'.extracted\'\n    if not marker.exists():\n        folder.mkdir(exist_ok=True)\n        with zipfile.ZipFile(target) as archive:\n            for item in archive.infolist():\n                path=(folder/item.filename).resolve()\n                if not path.is_relative_to(folder.resolve()):raise ValueError(\'Unsafe archive path\')\n            archive.extractall(folder)\n        marker.write_text(\'ok\',encoding=\'utf-8\')\n    report={\'source\':name,**spec,\'archive_sha256\':digest.hexdigest(),\'bytes\':target.stat().st_size,\n            \'images\':sum(p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\'} for p in folder.rglob(\'*\')),\n            \'status\':\'downloaded\',\'folder\':name}\n    (root/(name+\'-source.json\')).write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    print(json.dumps(report),flush=True);return report\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--sources\',nargs=\'+\',choices=SOURCES,default=[\'construction\',\'site_safety\',\'shwd\'])\n    p.add_argument(\'--root\',default=\'data/datasets/ppe-sources\');args=p.parse_args()\n    failures=[]\n    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:\n        futures={pool.submit(download_source,s,args.root):s for s in args.sources}\n        for f,name in [(f,futures[f]) for f in concurrent.futures.as_completed(futures)]:\n            try:f.result()\n            except Exception as exc:failures.append({\'source\':name,\'status\':\'failed\',\'error\':str(exc)})\n    Path(args.root,\'download-failures.json\').write_text(json.dumps(failures,indent=2),encoding=\'utf-8\')\n    return bool(failures)\n\n\nif __name__==\'__main__\':raise SystemExit(main())\n',encoding='utf-8')
(SCRIPTS/'prepare_ppe_data.py').write_text('"""Build separate, reproducible tasks without treating unlabelled PPE as absent.\n\nImages remain in ignored source folders. Prepared datasets contain real files\nand source/group manifests. Source test/validation images never enter training.\n"""\nimport argparse\nfrom collections import Counter,defaultdict\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport xml.etree.ElementTree as ET\n\nimport cv2\nimport numpy as np\nimport yaml\n\nSITE_NAMES=[\'Hardhat\',\'Mask\',\'NO-Hardhat\',\'NO-Mask\',\'NO-Safety Vest\',\'Person\',\n            \'Safety Cone\',\'Safety Vest\',\'machinery\',\'vehicle\']\nALIASES={\'helmet\':\'hardhat\',\'no_helmet\':\'no_hardhat\',\'bare_head\':\'no_hardhat\',\n         \'safety_vest\':\'vest\',\'no_safety_vest\':\'no_vest\'}\nTASKS={\'people\':[\'person\'],\'head_ppe\':[\'hardhat\',\'no_hardhat\'],\'vest_ppe\':[\'vest\',\'no_vest\']}\n\n\ndef crop_quality(crop):\n    """Conservative classifier visibility gate; this does not verify labels."""\n    h,w=crop.shape[:2]\n    if min(h,w)<24 or h*w<1024:return \'too_small\'\n    if not .2<=w/h<=4:return \'extreme_aspect\'\n    if cv2.Laplacian(cv2.cvtColor(crop,cv2.COLOR_BGR2GRAY),cv2.CV_64F).var()<15:\n        return \'blurred\'\n    return None\n\n\ndef normalized(s):\n    s=re.sub(\'[^a-z0-9]+\',\'_\',s.lower()).strip(\'_\')\n    return ALIASES.get(s,s)\n\n\ndef source_split(path):\n    parts=set(path.parts)\n    return \'test\' if \'test\' in parts else \'val\' if parts & {\'val\',\'valid\',\'validation\'} else \'train\'\n\n\ndef original_group(path):\n    # Roboflow augmentation variants must share a split, even with different pixels.\n    return re.sub(r\'(?:_jpg|_png)?\\.rf\\.[a-f0-9]+$\',\'\',path.stem,flags=re.I)\n\n\ndef records(root):\n    for source in sorted(p for p in Path(root).iterdir() if p.is_dir()):\n        if not (source/\'.extracted\').exists():raise ValueError(f\'Source extraction incomplete: {source}\')\n        configs=list(source.rglob(\'*.yaml\'))+list(source.rglob(\'*.yml\'))\n        names=None\n        for cfg in configs:\n            parsed=yaml.safe_load(cfg.read_text(encoding=\'utf-8\'))\n            if isinstance(parsed,dict) and parsed.get(\'names\'):\n                n=parsed[\'names\'];names={int(k):v for k,v in n.items()} if isinstance(n,dict) else dict(enumerate(n));break\n        if names is None and source.name==\'site_safety\':names=dict(enumerate(SITE_NAMES))\n        if names is None:\n            class_files=list(source.rglob(\'classes.txt\'))\n            if class_files:names=dict(enumerate(line.strip() for line in class_files[0].read_text().splitlines() if line.strip()))\n        annotations=list(source.rglob(\'Annotations\'))\n        xmls={p.stem:p for folder in annotations for p in folder.glob(\'*.xml\')}\n        native_splits={}\n        for split in [\'train\',\'val\',\'test\']:\n            for split_file in source.rglob(split+\'.txt\'):\n                if split_file.parent.name==\'Main\':\n                    for line in split_file.read_text().splitlines():\n                        if line.strip():native_splits[line.split()[0]]=split\n        for image in sorted(p for p in source.rglob(\'*\') if p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\'}):\n            labels=[]\n            if xmls:\n                xml=xmls.get(image.stem)\n                if xml is None:continue\n                node=ET.parse(xml).getroot();width=float(node.findtext(\'size/width\'));height=float(node.findtext(\'size/height\'))\n                for obj in node.findall(\'object\'):\n                    label=obj.findtext(\'name\');box=obj.find(\'bndbox\')\n                    # SHWD\'s "person" is an uncovered HEAD, not a person box.\n                    label=\'hardhat\' if label==\'hat\' else \'no_hardhat\' if label==\'person\' else normalized(label)\n                    a,b,c,d=[float(box.findtext(k)) for k in [\'xmin\',\'ymin\',\'xmax\',\'ymax\']]\n                    labels.append([label,(a+c)/2/width,(b+d)/2/height,(c-a)/width,(d-b)/height])\n                complete={\'head_ppe\'}\n            else:\n                if names is None:continue\n                relative=image.relative_to(source)\n                parts=list(relative.parts)\n                if \'images\' not in parts:continue\n                parts[parts.index(\'images\')]=\'labels\'\n                label_path=(source/Path(*parts)).with_suffix(\'.txt\')\n                if not label_path.exists():continue\n                for line in label_path.read_text().splitlines():\n                    fields=line.split()\n                    if len(fields)!=5:raise ValueError(f\'Expected detection boxes, not segmentation: {label_path}\')\n                    label=normalized(names[int(fields[0])]);box=list(map(float,fields[1:]))\n                    if not all(np.isfinite(box)) or not all(0<=v<=1 for v in box) or min(box[2:])<=0:\n                        raise ValueError(f\'Invalid bbox: {label_path}\')\n                    labels.append([label,*box])\n                source_classes={normalized(n) for n in names.values()}\n                complete={task for task,classes in TASKS.items() if set(classes)<=source_classes}\n                # This aggregate mixes several annotation schemes. Use only\n                # explicitly labelled positive crops until completeness and the\n                # meaning of head/person have been reviewed for each subsource.\n                if source.name==\'hardhat_vest\':complete=set()\n            yield {\'path\':str(image.resolve()),\'source\':source.name,\'split\':native_splits.get(image.stem,source_split(image)),\n                   \'group\':original_group(image),\'labels\':labels,\'complete_tasks\':sorted(complete)}\n\n\ndef prepare(root,output,summary_path,extra_roots=()):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if any(output.iterdir()):\n        raise ValueError(\'Output is not empty; choose a new output to avoid stale files\')\n    manifest=[];hashes={};groups={};buckets=defaultdict(list);duplicates=0;unusable=0\n    # Read held-out partitions first: duplicates and related variants in train\n    # inherit the held-out split. This keeps source test images out of train.\n    raw=[rec for source_root in [root,*extra_roots] for rec in records(source_root)]\n    raw.sort(key=lambda r:({\'test\':0,\'val\':1,\'train\':2}[r[\'split\']],-len(r[\'complete_tasks\']),r[\'path\']))\n    print(f\'Indexed {len(raw)} annotated source images\',flush=True)\n    for number,record in enumerate(raw):\n        if number%1000==0:print(f\'Deduplicating {number}/{len(raw)}\',flush=True)\n        if not record[\'complete_tasks\'] and not any(v[0] in {\'hardhat\',\'vest\'} for v in record[\'labels\']):\n            unusable+=1;continue\n        path=Path(record[\'path\']);data=path.read_bytes();digest=hashlib.sha256(data).hexdigest()\n        image=cv2.imdecode(np.frombuffer(data,np.uint8),cv2.IMREAD_COLOR)\n        if image is None:raise ValueError(f\'Unreadable image: {path}\')\n        gray=cv2.resize(cv2.cvtColor(image,cv2.COLOR_BGR2GRAY),(32,32)).astype(np.float32)\n        low=cv2.dct(gray)[:8,:8].ravel();bits=low>np.median(low[1:])\n        phash=sum(int(v)<<i for i,v in enumerate(bits))\n        candidates=set(i for k in range(8) for i in buckets[(k,(phash>>(k*8))&255)])\n        near=next((i for i in candidates if (manifest[i][\'phash\']^phash).bit_count()<=2),None)\n        duplicate=hashes.get(digest,near)\n        if duplicate is not None:\n            duplicates+=1\n            previous=manifest[duplicate]\n            groups[(record[\'source\'],record[\'group\'])]=previous[\'split\']\n            if len(record[\'complete_tasks\'])>len(previous[\'complete_tasks\']):\n                record.update(split=previous[\'split\'],sha256=digest,phash=phash,width=image.shape[1],height=image.shape[0])\n                manifest[duplicate]=record\n            hashes[digest]=duplicate\n            continue\n        # Group keys may over-group coincident source names; that is conservative\n        # for leakage. Exact/perceptual matches are handled globally above.\n        group=(record[\'source\'],record[\'group\'])\n        split=groups.setdefault(group,record[\'split\'])\n        if split==\'train\':\n            bucket=int(hashlib.sha256(\'|\'.join(group).encode()).hexdigest()[:8],16)%100\n            split=\'val\' if bucket<10 else \'test\' if bucket<20 else \'train\'\n        record.update(split=split,sha256=digest,phash=phash,width=image.shape[1],height=image.shape[0])\n        hashes[digest]=len(manifest)\n        for k in range(8):buckets[(k,(phash>>(k*8))&255)].append(len(manifest))\n        manifest.append(record)\n    rank={\'test\':0,\'val\':1,\'train\':2}\n    for rec in manifest:\n        group_split=groups.get((rec[\'source\'],rec[\'group\']),rec[\'split\'])\n        if rank[group_split]<rank[rec[\'split\']]:rec[\'split\']=group_split\n    task_counts={};crop_counts=defaultdict(Counter)\n    for task,classes in TASKS.items():\n        print(f\'Writing {task}\',flush=True)\n        counts=Counter();folder=output/task\n        for rec in manifest:\n            name=rec[\'sha256\'];split=rec[\'split\'];src=Path(rec[\'path\'])\n            selected=[v for v in rec[\'labels\'] if v[0] in classes]\n            if task in rec[\'complete_tasks\']:\n                im=folder/\'images\'/split/(name+src.suffix.lower());lab=folder/\'labels\'/split/(name+\'.txt\')\n                im.parent.mkdir(parents=True,exist_ok=True);lab.parent.mkdir(parents=True,exist_ok=True)\n                if not im.exists():shutil.copyfile(src,im)\n                lab.write_text(\'\\n\'.join(str(classes.index(v[0]))+\' \'+\' \'.join(map(str,v[1:])) for v in selected)+\'\\n\',encoding=\'utf-8\')\n                counts[split]+=1\n            if task in {\'head_ppe\',\'vest_ppe\'} and selected:\n                frame=cv2.imread(str(src));h,w=frame.shape[:2]\n                for i,(label,cx,cy,bw,bh) in enumerate(selected):\n                    # Context is needed around a helmet box; no synthetic\n                    # "unknown" examples or missing-annotation negatives.\n                    x1,y1=max(0,int((cx-bw*.65)*w)),max(0,int((cy-bh*.65)*h))\n                    x2,y2=min(w,int((cx+bw*.65)*w)),min(h,int((cy+bh*.65)*h))\n                    if min(x2-x1,y2-y1)<16:continue\n                    dest=output/(task+\'_classifier\')/split/label/(name+f\'-{i}.jpg\')\n                    dest.parent.mkdir(parents=True,exist_ok=True)\n                    crop=frame[y1:y2,x1:x2]\n                    if crop_quality(crop):continue\n                    cv2.imwrite(str(dest),crop);crop_counts[task][split+\':\'+label]+=1\n        folder.mkdir(parents=True,exist_ok=True)\n        config={\'path\':str(folder.resolve()),\'train\':\'images/train\',\'val\':\'images/val\',\'test\':\'images/test\',\'names\':dict(enumerate(classes))}\n        (folder/\'data.yaml\').write_text(yaml.safe_dump(config,sort_keys=False),encoding=\'utf-8\')\n        task_counts[task]=dict(counts)\n    with (output/\'manifest.jsonl\').open(\'w\',encoding=\'utf-8\') as f:\n        for rec in manifest:f.write(json.dumps(rec)+\'\\n\')\n    summary={\'raw_images\':len(raw),\'unique_images\':len(manifest),\'duplicates_removed\':duplicates,\n             \'excluded_incomplete_annotation_images\':unusable,\n             \'sources\':dict(Counter(r[\'source\'] for r in manifest)),\'splits\':dict(Counter(r[\'split\'] for r in manifest)),\n             \'tasks\':task_counts,\'classification_crops\':{k:dict(v) for k,v in crop_counts.items()},\n             \'baseline_construction_train_images\':1132,\'upstream_claimed_train_images\':68253,\n             \'upstream_4x_train_target\':273012,\'actual_unique_train_images\':sum(r[\'split\']==\'train\' for r in manifest),\n             \'upstream_4x_reached\':sum(r[\'split\']==\'train\' for r in manifest)>=273012,\n             \'limitations\':[\'Source group metadata may be incomplete; camera-disjoint independence is not established.\',\n                            \'No unobservable-class crops are fabricated; manual visibility labels remain required.\',\n                            \'Head and torso classifier crops need human label/geometry review before a full training run.\']}\n    Path(summary_path).parent.mkdir(parents=True,exist_ok=True)\n    Path(summary_path).write_text(json.dumps(summary,indent=2)+\'\\n\',encoding=\'utf-8\')\n    try:\n        from scripts.finalize_ppe_data import finalize\n    except ModuleNotFoundError:\n        from finalize_ppe_data import finalize\n    summary=finalize(output,summary_path,output.parent/\'ppe-excluded-manifest.jsonl\')\n    print(json.dumps(summary,indent=2));return summary\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',default=\'data/datasets/ppe-sources\')\n    p.add_argument(\'--extra-root\',action=\'append\',default=[])\n    p.add_argument(\'--output\',default=\'data/datasets/ppe-prepared\');p.add_argument(\'--summary\',default=\'docs/ppe-dataset-summary.json\')\n    a=p.parse_args();prepare(a.root,a.output,a.summary,a.extra_root)\n',encoding='utf-8')
(SCRIPTS/'finalize_ppe_data.py').write_text('"""Keep only source images that contribute detector examples or usable crops."""\nimport argparse\nfrom collections import Counter\nimport json\nfrom pathlib import Path\n\n\ndef finalize(data,summary_path,excluded):\n    data=Path(data);summary_path=Path(summary_path);excluded=Path(excluded)\n    summary=json.loads(summary_path.read_text());manifest=data/\'manifest.jsonl\'\n    rows=[json.loads(line) for line in manifest.read_text().splitlines()]\n    used={p.stem for task in [\'people\',\'head_ppe\',\'vest_ppe\'] for p in (data/task/\'images\').rglob(\'*\') if p.is_file()}\n    used.update(p.stem[:64] for task in [\'head_ppe_classifier\',\'vest_ppe_classifier\'] for p in (data/task).rglob(\'*.jpg\'))\n    kept=[r for r in rows if r[\'sha256\'] in used];removed=[r for r in rows if r[\'sha256\'] not in used]\n    summary.setdefault(\'unique_images_before_crop_filter\',summary[\'unique_images\'])\n    summary.setdefault(\'splits_before_crop_filter\',summary[\'splits\'])\n    summary[\'unique_images\']=len(kept)\n    summary[\'excluded_images_without_usable_crops\']=summary[\'unique_images_before_crop_filter\']-len(kept)\n    summary[\'sources\']=dict(Counter(r[\'source\'] for r in kept))\n    summary[\'splits\']=dict(Counter(r[\'split\'] for r in kept))\n    summary[\'actual_unique_train_images\']=summary[\'splits\'].get(\'train\',0)\n    summary[\'upstream_4x_reached\']=summary[\'actual_unique_train_images\']>=summary[\'upstream_4x_train_target\']\n    if removed:\n        excluded.parent.mkdir(parents=True,exist_ok=True)\n        with excluded.open(\'a\',encoding=\'utf-8\') as f:\n            for row in removed:f.write(json.dumps(row)+\'\\n\')\n        temporary=manifest.with_suffix(\'.tmp\')\n        with temporary.open(\'w\',encoding=\'utf-8\') as f:\n            for row in kept:f.write(json.dumps(row)+\'\\n\')\n        temporary.replace(manifest)\n    summary_path.write_text(json.dumps(summary,indent=2)+\'\\n\')\n    print(\'Contributing unique images:\',len(kept),\'train:\',summary[\'actual_unique_train_images\'],flush=True)\n    return summary\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',default=\'data/datasets/ppe-expanded\')\n    p.add_argument(\'--summary\',default=\'docs/ppe-dataset-summary.json\')\n    p.add_argument(\'--excluded\',default=\'data/datasets/ppe-rejected-crops/unused-manifest.jsonl\')\n    a=p.parse_args();finalize(a.data,a.summary,a.excluded)\n',encoding='utf-8')
(SCRIPTS/'train_ppe.py').write_text('"""Train one PPE task; test is evaluated only after validation selects weights."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef train_detector(args):\n    from ultralytics import YOLO,settings\n    # Disable optional account integrations only in this process. Do not write\n    # the user\'s persistent Ultralytics settings on a local training smoke.\n    dict.update(settings,{key:False for key in [\'wandb\',\'mlflow\',\'comet\',\'clearml\',\'neptune\',\'raytune\',\'hub\'] if key in settings})\n    import yaml\n    output=Path(args.output);output.mkdir(parents=True,exist_ok=True)\n    config=yaml.safe_load((Path(args.data)/args.task/\'data.yaml\').read_text())\n    config[\'path\']=str((Path(args.data)/args.task).resolve())\n    data_file=output/(args.task+\'-data.yaml\');data_file.write_text(yaml.safe_dump(config),encoding=\'utf-8\')\n    last=output/args.task/\'weights/last.pt\'\n    model=YOLO(str(last) if args.resume and last.exists() else args.weights)\n    model.train(data=str(data_file),epochs=args.epochs,\n                imgsz=args.imgsz,batch=args.batch,device=args.device,seed=42,\n                project=args.output,name=args.task,exist_ok=True,patience=30,\n                close_mosaic=min(10,args.epochs),workers=2,amp=True,save_period=5,\n                resume=bool(args.resume and last.exists()))\n    best=Path(args.output)/args.task/\'weights/best.pt\'\n    model=YOLO(str(best))\n    metrics=model.val(data=str(data_file),split=\'test\',\n                      imgsz=args.imgsz,batch=args.batch,device=args.device,\n                      project=str(output),name=args.task+\'-test\',exist_ok=True)\n    exported=Path(model.export(format=\'onnx\',imgsz=args.imgsz,batch=1,dynamic=False,\n                               simplify=False,opset=12,nms=False,device=\'cpu\'))\n    deployment={\'people\':\'person-yolo12m.onnx\',\'head_ppe\':\'ppe-head.onnx\',\'vest_ppe\':\'ppe-vest.onnx\'}[args.task]\n    import shutil\n    target=Path(args.output)/deployment;shutil.copyfile(exported,target)\n    if args.task==\'people\':shutil.copyfile(best,Path(args.output)/\'person-yolo12m.pt\')\n    result={\'task\':args.task,\'test_metrics\':{k:float(v) for k,v in metrics.results_dict.items()},\n            \'onnx_sha256\':hashlib.sha256(target.read_bytes()).hexdigest(),\n            \'linq_status\':\'not_compiled\',\'event_metrics\':\'not_evaluated\'}\n    (Path(args.output)/(args.task+\'-metrics.json\')).write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n\n\ndef train_classifier(args):\n    import numpy as np\n    import torch\n    from torch.utils.data import DataLoader\n    from torchvision import datasets,models,transforms\n    import onnx\n    torch.manual_seed(42)\n    root=Path(args.data)/(args.task+\'_classifier\')\n    mean=[.485,.456,.406];std=[.229,.224,.225]\n    common=[transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean,std)]\n    train=datasets.ImageFolder(root/\'train\',transforms.Compose([\n        transforms.Resize((224,224)),transforms.RandomHorizontalFlip(),\n        transforms.ColorJitter(.15,.15,.15,.03),*common[1:]]))\n    val=datasets.ImageFolder(root/\'val\',transforms.Compose(common))\n    test=datasets.ImageFolder(root/\'test\',transforms.Compose(common))\n    if train.class_to_idx!=val.class_to_idx or train.class_to_idx!=test.class_to_idx:\n        raise ValueError(\'Every split must have the same labelled classes\')\n    if min(len(train),len(val),len(test))<20:raise ValueError(\'Not enough reviewed crops for independent splits\')\n    device=torch.device(\'cuda:\'+args.device if str(args.device).isdigit() else args.device)\n    if args.classifier==\'resnet152\':\n        model=models.resnet152(weights=models.ResNet152_Weights.DEFAULT)\n    else:model=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)\n    model.fc=torch.nn.Linear(model.fc.in_features,len(train.classes));model.to(device)\n    loaders={k:DataLoader(d,batch_size=args.batch,shuffle=k==\'train\',num_workers=2)\n             for k,d in [(\'train\',train),(\'val\',val),(\'test\',test)]}\n    weights=np.bincount(train.targets,minlength=len(train.classes))\n    loss_fn=torch.nn.CrossEntropyLoss(weight=torch.tensor(1/np.maximum(weights,1),dtype=torch.float32,device=device))\n    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)\n    folder=Path(args.output);folder.mkdir(parents=True,exist_ok=True)\n    best_loss=float(\'inf\');checkpoint=folder/(args.task+\'-classifier.pt\')\n    start_epoch=0;last=folder/(args.task+\'-classifier-last.pt\')\n    if args.resume and last.exists():\n        state=torch.load(last,map_location=device,weights_only=True)\n        if state[\'classes\']!=train.classes:raise ValueError(\'Resume class order differs\')\n        model.load_state_dict(state[\'state_dict\']);optimizer.load_state_dict(state[\'optimizer\'])\n        start_epoch=state[\'epoch\'];best_loss=state[\'best_loss\']\n    def evaluate(loader):\n        model.eval();losses=[];scores=[];targets=[]\n        with torch.no_grad():\n            for x,y in loader:\n                x,y=x.to(device),y.to(device);logits=model(x)\n                losses.append(float(loss_fn(logits,y)));scores.extend(logits.softmax(1).cpu().numpy());targets.extend(y.cpu().numpy())\n        return float(np.mean(losses)),np.asarray(scores),np.asarray(targets)\n    for epoch in range(start_epoch,args.epochs):\n        model.train()\n        for x,y in loaders[\'train\']:\n            optimizer.zero_grad(set_to_none=True);loss=loss_fn(model(x.to(device)),y.to(device))\n            loss.backward();optimizer.step()\n        validation_loss,_,_=evaluate(loaders[\'val\'])\n        print(json.dumps({\'epoch\':epoch+1,\'val_loss\':validation_loss}),flush=True)\n        if validation_loss<best_loss:\n            best_loss=validation_loss;torch.save({\'state_dict\':model.state_dict(),\'classes\':train.classes},checkpoint)\n        torch.save({\'state_dict\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\n                    \'epoch\':epoch+1,\'best_loss\':best_loss,\'classes\':train.classes},last)\n    saved=torch.load(checkpoint,map_location=device,weights_only=True);model.load_state_dict(saved[\'state_dict\'])\n    _,probabilities,truth=evaluate(loaders[\'val\'])\n    choices=[]\n    for threshold in np.linspace(.5,.99,50):\n        selected=probabilities.max(1)>=threshold;pred=probabilities.argmax(1)\n        per_class=[]\n        for label in range(len(train.classes)):\n            positives=selected & (pred==label);count=int(positives.sum())\n            per_class.append((count,int(((truth==label)&positives).sum())/max(count,1)))\n        if all(count>=20 and precision>=.95 for count,precision in per_class):\n            choices.append((float(selected.mean()),float(threshold)))\n    threshold=max(choices)[1] if choices else .8\n    _,test_scores,test_truth=evaluate(loaders[\'test\']);test_pred=test_scores.argmax(1)\n    # Rejections count as misses for recall on assessable test crops.\n    selected=test_scores.max(1)>=threshold;stats={}\n    for label,name in enumerate(train.classes):\n        tp=int(((test_pred==label)&(test_truth==label)&selected).sum())\n        fp=int(((test_pred==label)&(test_truth!=label)&selected).sum())\n        fn=int(((test_truth==label)&((test_pred!=label)|~selected)).sum())\n        precision=tp/max(tp+fp,1);recall=tp/max(tp+fn,1)\n        stats[name]={\'precision\':precision,\'recall\':recall,\'f1\':2*precision*recall/max(precision+recall,1e-12),\'tp\':tp,\'fp\':fp,\'fn\':fn}\n    name=\'ppe-head-classifier.onnx\' if args.task==\'head_ppe\' else \'ppe-torso-classifier.onnx\'\n    target=folder/name;model.eval().cpu()\n    torch.onnx.export(model,torch.zeros(1,3,224,224),str(target),input_names=[\'images\'],\n                      output_names=[\'logits\'],opset_version=12,dynamo=False)\n    graph=onnx.load(target)\n    for key,value in {\'class_names\':json.dumps(train.classes),\'rgb_mean\':json.dumps(mean),\'rgb_std\':json.dumps(std),\n                      \'abstain_threshold\':str(threshold),\'crop_policy\':\'proposal_context_1.3\',\n                      \'resize_algorithm\':\'pil_bilinear\'}.items():\n        item=graph.metadata_props.add();item.key=key;item.value=value\n    onnx.save(graph,target)\n    report={\'classes\':train.classes,\'threshold\':threshold,\'validation_precision_target_met\':bool(choices),\n            \'test\':stats,\'test_coverage\':float(selected.mean()),\'linq_status\':\'not_compiled\',\n            \'warning\':\'Crop classification metrics are not event metrics; unobservable crops need a separate evaluation set.\'}\n    (folder/(args.task+\'-classifier-metrics.json\')).write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',default=\'data/datasets/ppe-prepared\')\n    p.add_argument(\'--task\',choices=[\'people\',\'head_ppe\',\'vest_ppe\'],required=True)\n    p.add_argument(\'--mode\',choices=[\'detect\',\'classify\'],default=\'detect\')\n    p.add_argument(\'--weights\',default=\'yolo12m.pt\');p.add_argument(\'--classifier\',choices=[\'resnet50\',\'resnet152\'],default=\'resnet50\')\n    p.add_argument(\'--epochs\',type=int,default=150);p.add_argument(\'--imgsz\',type=int,default=640)\n    p.add_argument(\'--batch\',type=int,default=4);p.add_argument(\'--device\',default=\'0\')\n    p.add_argument(\'--resume\',action=\'store_true\')\n    p.add_argument(\'--output\',default=\'training-output\');a=p.parse_args()\n    if a.mode==\'classify\' and a.task==\'people\':p.error(\'Classifier is only for head/torso PPE\')\n    (train_detector if a.mode==\'detect\' else train_classifier)(a)\n',encoding='utf-8')
(SCRIPTS/'check_linq_compatibility.py').write_text('"""Check the documented YOLO12m split and optionally compile it with LinQ SDK.\n\nNo compatibility claim is made from ONNX checks alone. Run --compile only in\nthe separate Linux/Python 3.12 SDK environment. Calibration is train-only NPZ.\n"""\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\n\nimport numpy as np\nimport onnx\nimport onnxruntime as ort\n\nCUTS = [\'/model.21/Div_1_output_0\', \'/model.21/Sub_1_output_0\',\n        \'/model.21/Sigmoid_output_0\']\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef preflight(model, output, images=()):\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    graph = onnx.load(str(model)); onnx.checker.check_model(graph)\n    inputs = [v.name for v in graph.graph.input]\n    outputs = [v.name for v in graph.graph.output]\n    available = {v for n in graph.graph.node for v in n.output}\n    if not set(CUTS) <= available:\n        raise ValueError(\'This export differs from the documented YOLO12m graph: missing cut tensors\')\n    dims = [[d.dim_value for d in v.type.tensor_type.shape.dim] for v in graph.graph.input]\n    if len(inputs) != 1 or any(d <= 0 for shape in dims for d in shape):\n        raise ValueError(\'One static input is required\')\n    head, tail = output/\'yolo12m-head.onnx\', output/\'yolo12m-tail.onnx\'\n    onnx.utils.extract_model(str(model), str(head), inputs, CUTS)\n    onnx.utils.extract_model(str(model), str(tail), CUTS, outputs)\n    options = ort.SessionOptions(); options.intra_op_num_threads = 4\n    sessions = [ort.InferenceSession(str(p), options, providers=[\'CPUExecutionProvider\'])\n                for p in (model, head, tail)]\n    # Random data checks graph wiring only; it is NEVER calibration data.\n    tensors = [np.random.default_rng(42).random(dims[0], dtype=np.float32)]\n    if images:\n        import cv2\n        from app.scenario_models import letterbox_tensor\n        for path in images:\n            image = cv2.imread(str(path))\n            if image is None: raise ValueError(f\'Cannot read {path}\')\n            tensors.append(letterbox_tensor(image, (dims[0][3], dims[0][2]))[0])\n    parity = []\n    for tensor in tensors:\n        full = sessions[0].run(None, {inputs[0]: tensor})\n        intermediate = sessions[1].run(None, {inputs[0]: tensor})\n        restored = sessions[2].run(None, dict(zip(CUTS, intermediate)))\n        for actual, expected in zip(restored, full):\n            np.testing.assert_allclose(actual, expected, rtol=1e-5, atol=1e-4)\n        parity.append(max(float(np.max(np.abs(a-b))) for a,b in zip(full, restored)))\n    report = {\'model_sha256\': sha(model), \'onnx_ir\': graph.ir_version,\n              \'opsets\': {x.domain: x.version for x in graph.opset_import},\n              \'input_names\': inputs, \'input_shapes\': dims, \'cut_tensors\': CUTS,\n              \'head_sha256\': sha(head), \'tail_sha256\': sha(tail),\n              \'parity_max_abs_errors\': parity, \'parity_samples\': len(tensors),\n              \'real_image_samples\': len(tensors)-1, \'status\': \'onnx_split_verified\',\n              \'sdk_compiled\': False, \'hardware_verified\': False,\n              \'warning\': \'ONNX split parity is not SDK compilation, hardware parity or accuracy evaluation.\'}\n    return report\n\n\ndef compile_head(output, calibration):\n    if platform.system() != \'Linux\' or sys.version_info[:2] != (3,12):\n        raise RuntimeError(\'LinQ binary SDK needs Linux x86_64 / Python 3.12\')\n    from tpu_framework import RegularModel, Network, compiler, TPU_H1M, O1, TpuProgram\n    from tpu_framework import preliminary_model_conditioning\n    from dnn_quant.common import SupportedLayerTypes as Slt\n    from dnn_quant.autoquant.cfg_wrappers.cfg_quant_block_wrappers import FLOAT16, PIECEWISE_LUT\n    data = np.load(calibration, allow_pickle=False)\n    if \'images\' not in data or \'split\' not in data or str(data[\'split\'].item()) != \'train\':\n        raise ValueError(\'Calibration NPZ must contain images and split="train"\')\n    images = data[\'images\'].astype(np.float32)\n    graph = preliminary_model_conditioning(onnx.load(str(output/\'yolo12m-head.onnx\')))\n    shape = [d.dim_value for d in graph.graph.input[0].type.tensor_type.shape.dim]\n    if images.ndim != 4 or list(images.shape[1:]) != shape[1:] or len(images) < 8:\n        raise ValueError(\'Need >=8 preprocessed train samples matching the static CHW input\')\n    if not np.isfinite(images).all() or images.min()<0 or images.max()>1:\n        raise ValueError(\'Calibration requires finite RGB /255 letterboxed tensors\')\n    regular = RegularModel(model=graph, operation_cfg={Slt.FW_LAYER_REDUCE_MAX: FLOAT16,\n                           Slt.FW_LAYER_REDUCE_MEAN: FLOAT16, Slt.FW_LAYER_REDUCE_SUM: FLOAT16},\n                           swish_layer_mode=PIECEWISE_LUT)\n    name = graph.graph.input[0].name\n    name = regular.anchors_mapping.get(name, name)\n    thresholds = regular.calibrate_torch(calibration_data={name: images}, device=\'cpu\')\n    regular.dump_thresholds(str(output/\'thresholds.json\'), thresholds)\n    quantized = regular.quantize(thresholds)\n    quantized.save(str(output), \'yolo12m-head.qm.onnx\')\n    network, _ = Network.from_quantized_model(quantized)\n    executable, descriptions = compiler.compile_(hardware_parameters=TPU_H1M, network=network, parameters=O1)\n    target = output/\'yolo12m-head-b1-o1-h1m.tpu\'\n    TpuProgram.from_executable(executable, descriptions).to_file(str(target))\n    return {\'sdk_compiled\': True, \'status\': \'compiled_unverified\', \'program_sha256\': sha(target),\n            \'calibration_samples\': len(images), \'calibration_sha256\': sha(calibration),\n            \'hardware_verified\': False}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--model\', default=\'data/models/yolo12m.onnx\')\n    parser.add_argument(\'--output\', default=\'data/linq/yolo12m\')\n    parser.add_argument(\'--report\', default=\'docs/yolo12m-compatibility.json\')\n    parser.add_argument(\'--image\', action=\'append\', default=[])\n    parser.add_argument(\'--compile\', action=\'store_true\')\n    parser.add_argument(\'--calibration\')\n    args = parser.parse_args()\n    report = preflight(args.model, args.output, args.image)\n    code = 0\n    if args.compile:\n        try:\n            if not args.calibration: raise ValueError(\'--calibration is required\')\n            report.update(compile_head(Path(args.output), args.calibration))\n        except Exception as exc:\n            report.update(status=\'compile_failed\', compile_error=f\'{type(exc).__name__}: {exc}\')\n            code = 1\n    Path(args.report).parent.mkdir(parents=True, exist_ok=True)\n    Path(args.report).write_text(json.dumps(report, indent=2)+\'\\n\', encoding=\'utf-8\')\n    print(json.dumps(report, indent=2)); return code\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n',encoding='utf-8')

In [ ]:
MODE='smoke'  # После проверки смените на 'full'; smoke не является обученной финальной моделью.
RESUME_FROM=None  # Например Path('/kaggle/input/my-previous-checkpoints/training-output')
SOURCES=['construction','site_safety','shwd','hardhat_vest']
# 'sh17' (~14 GB) добавляйте только с учётом CC BY-NC-SA 4.0 и свободного диска.
# HardHat-Vest: только явные положительные crop; head/person не означают отрицательные СИЗ.
EPOCHS=1 if MODE=='smoke' else 150
CLASSIFIER_EPOCHS=1 if MODE=='smoke' else 30
LABELS_REVIEWED=False  # В full ставить True после проверки разметки и crop, не по результатам smoke.
IMGSZ=640  # Сначала статический 640 для воспроизведения LinQ; 960/1280 - отдельная проверка.
BATCH=4
OUTPUT=WORK/('smoke-output' if MODE=='smoke' else 'training-output')
OUTPUT.mkdir(exist_ok=True)
if RESUME_FROM: shutil.copytree(RESUME_FROM,OUTPUT,dirs_exist_ok=True)

In [ ]:
input_root=Path('/kaggle/input')
manifests=list(input_root.rglob('manifest.jsonl'))
archives=list(input_root.rglob('linq-ppe-training-data.zip'))
if manifests:
    DATA=manifests[0].parent
elif archives:
    DATA=WORK/'ppe-prepared'
    with zipfile.ZipFile(archives[0]) as z:
        for member in z.infolist():
            if not (DATA/member.filename).resolve().is_relative_to(DATA.resolve()):
                raise ValueError('Unsafe dataset archive path')
        z.extractall(DATA)
else:
    DATA=WORK/'ppe-prepared'; RAW=WORK/'ppe-sources'
    subprocess.run([sys.executable,str(SCRIPTS/'download_ppe_data.py'),
                    '--root',str(RAW),'--sources',*SOURCES],check=True)
    subprocess.run([sys.executable,str(SCRIPTS/'prepare_ppe_data.py'),
                    '--root',str(RAW),'--output',str(DATA),
                    '--summary',str(OUTPUT/'dataset-summary.json')],check=True)
for task in ['people','head_ppe','vest_ppe']:
    assert (DATA/task/'data.yaml').exists(), f'Не найден dataset {task}'
print('Dataset:', DATA)

Перед полным запуском просмотрите случайные crop и проверьте классы. Особенно важны
`no_hardhat` и `no_vest`. Слишком маленькая или закрытая область должна попадать
в отдельный вручную размеченный набор видимости. Notebook не создаёт фиктивный
класс unknown. До появления такого набора используется отказ по порогу уверенности.
Автоматически исключаются crop меньше 24 px по стороне / 1024 px по площади,
крайние пропорции и сильное размытие. Это не проверяет правильность исходной разметки.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
rng=np.random.default_rng(42)
paths=list((DATA/'head_ppe_classifier/train').rglob('*.jpg'))+list((DATA/'vest_ppe_classifier/train').rglob('*.jpg'))
sample=rng.choice(len(paths),min(16,len(paths)),replace=False)
fig,axes=plt.subplots(4,4,figsize=(12,12))
for ax,index in zip(axes.flat,sample):
    p=paths[int(index)];ax.imshow(Image.open(p));ax.set_title(p.parent.name);ax.axis('off')
plt.tight_layout()

In [ ]:
assert MODE=='smoke' or LABELS_REVIEWED, 'Просмотрите разметку и crop, затем задайте LABELS_REVIEWED=True.'
if MODE=='smoke':
    # A bounded pipeline test, never a quality benchmark or release model.
    FULL_DATA=DATA; DATA=WORK/'smoke-data'
    for task in ['people','head_ppe','vest_ppe']:
        target=DATA/task;target.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(FULL_DATA/task/'data.yaml',target/'data.yaml')
        for split,limit in [('train',48),('val',16),('test',16)]:
            for image in sorted((FULL_DATA/task/'images'/split).glob('*'))[:limit]:
                for kind,source in [('images',image),('labels',FULL_DATA/task/'labels'/split/(image.stem+'.txt'))]:
                    dest=target/kind/split/source.name;dest.parent.mkdir(parents=True,exist_ok=True)
                    shutil.copyfile(source,dest)
    for task in ['head_ppe_classifier','vest_ppe_classifier']:
        for split,limit in [('train',48),('val',16),('test',16)]:
            for classdir in sorted((FULL_DATA/task/split).iterdir()):
                for source in sorted(classdir.glob('*.jpg'))[:limit]:
                    dest=DATA/task/split/classdir.name/source.name;dest.parent.mkdir(parents=True,exist_ok=True)
                    shutil.copyfile(source,dest)
    print('Smoke uses small fixed subsets:',DATA)

In [ ]:
for task in ['people','head_ppe','vest_ppe']:
    subprocess.run([sys.executable,str(SCRIPTS/'train_ppe.py'),'--data',str(DATA),
                    '--task',task,'--epochs',str(EPOCHS),'--imgsz',str(IMGSZ),
                    '--batch',str(BATCH),'--device','0','--output',str(OUTPUT),'--resume'],check=True)

In [ ]:
for task in ['head_ppe','vest_ppe']:
    subprocess.run([sys.executable,str(SCRIPTS/'train_ppe.py'),'--data',str(DATA),
                    '--task',task,'--mode','classify','--classifier','resnet152',
                    '--epochs',str(CLASSIFIER_EPOCHS),'--batch','16','--device','0',
                    '--output',str(OUTPUT),'--resume'],check=True)

In [ ]:
# Train-only representative calibration tensors. No test images enter this file.
import cv2
for task in ['people','head_ppe','vest_ppe']:
    files=sorted((DATA/task/'images/train').glob('*'))
    rng=np.random.default_rng(42);chosen=rng.choice(len(files),min(300,len(files)),replace=False)
    tensor=[]
    for index in chosen:
        image=cv2.imread(str(files[int(index)]));h,w=image.shape[:2]
        scale=min(IMGSZ/w,IMGSZ/h);nw,nh=round(w*scale),round(h*scale)
        left,top=round((IMGSZ-nw)/2-.1),round((IMGSZ-nh)/2-.1)
        padded=np.full((IMGSZ,IMGSZ,3),114,np.uint8)
        padded[top:top+nh,left:left+nw]=cv2.resize(image,(nw,nh))
        tensor.append(padded[:,:,::-1].transpose(2,0,1).astype(np.float32)/255)
    np.savez_compressed(OUTPUT/f'calibration-{task}-train.npz',images=np.stack(tensor),split=np.array('train'))
    del tensor
# Split parity is local ONNX verification; SDK compilation is performed separately.
subprocess.run([sys.executable,str(SCRIPTS/'check_linq_compatibility.py'),
                '--model',str(OUTPUT/'person-yolo12m.onnx'),'--output',str(OUTPUT/'linq-person'),
                '--report',str(OUTPUT/'linq-preflight.json')],check=True)

После полного обучения скачайте `linq-trained-models.zip`. Сначала изучите metrics JSON
и результаты на независимых видео. Детекторные mAP и crop F1 не заменяют event F1.
Для LinQ перенесите ONNX и train-калибровку в Linux 3.12 с SDK 2026.9.45, выполните
`check_linq_compatibility.py --compile --calibration ...`, затем проверьте `.tpu` на устройстве.
После обучения меняется число классов: разрез графа снова проверяется, совместимость
предобученного COCO-графа автоматически на новый checkpoint не переносится.

Для локального сайта: проверенные `person-yolo12m.pt`, `ppe-head.onnx`, `ppe-vest.onnx`,
`ppe-head-classifier.onnx`, `ppe-torso-classifier.onnx` положить в `data/models` и перезапустить.
Старые веса и отчёты сохраните для отката. Smoke-артефакты в рабочий сайт не устанавливать.

In [ ]:
manifest={'mode':MODE,'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0),
          'linq_hardware_verified':False,'event_metrics':'not_evaluated',
          'warning':'A smoke run is a pipeline check, not a release model.'}
(OUTPUT/'run-manifest.json').write_text(json.dumps(manifest,indent=2))
shutil.make_archive(str(WORK/'linq-trained-models'),'zip',OUTPUT)
print('Скачать:',WORK/'linq-trained-models.zip')